In [2]:
# -*- coding: GBK -*-
import json
import pandas as pd
import folium
from folium import plugins
import datetime
import sys
import requests






In [3]:
# 基本读取
response = requests.get('https://brouter.de/brouter?lonlats=120.61348599,31.33408267|121.31566408,31.19720282&profile=rail&format=geojson&alternativeidx=0')

print(response.json())  # JSON 内容

{'type': 'FeatureCollection', 'features': [{'type': 'Feature', 'properties': {'creator': 'BRouter-1.7.9', 'name': 'brouter_rail_0', 'track-length': '77514', 'filtered ascend': '2', 'plain-ascend': '-1', 'total-time': '6201', 'total-energy': '3407852', 'cost': '77514', 'messages': [['Longitude', 'Latitude', 'Elevation', 'Distance', 'CostPerKm', 'ElevCost', 'TurnCost', 'NodeCost', 'InitialCost', 'WayTags', 'NodeTags', 'Time', 'Energy'], ['120614191', '31333621', '6', '54', '1000', '0', '0', '0', '0', 'reversedirection=yes railway=rail', 'railway=switch', '4', '2283'], ['120615308', '31333840', '5', '109', '1000', '0', '0', '0', '0', 'reversedirection=yes railway=rail', 'railway=switch', '13', '6630'], ['120616266', '31333980', '3', '92', '1000', '0', '0', '0', '0', 'railway=rail', 'railway=switch', '20', '9673'], ['120699882', '31341148', '5', '8036', '1000', '0', '0', '0', '0', 'railway=rail', '', '663', '364243'], ['120710384', '31343543', '6', '1034', '1000', '0', '0', '0', '0', 'reve

In [32]:
database = "../../database"
output = ""

print("reading station data")
statdata = pd.read_csv(f'{database}//stations_data.csv',encoding='gbk')
stat_affiliation = {}
statgeo = {}
for i,row in statdata.iterrows():
    lat = row['纬度']
    lng = row['经度']
    loc = [lat,lng]
#     loc = correct(loc)
    statgeo[row['车站']]=loc
    stat_affiliation[row['车站']]={'lat':lat,'lng':lng,
                                 'bureau':row['路局'],
                                 'city':row['市'],
                                 'province':row['省']}
#print(statgeo)    

def generate_offsets(step=0.01, dimensions=4):
    """
    生成沿各个坐标轴正负方向的偏移向量
    
    Args:
        step: 偏移步长
        dimensions: 维度数量
    
    Returns:
        list: 包含所有偏移向量的列表
    """
    offsets = []
    for dim in range(dimensions):
        # 负方向
        neg = [0] * dimensions
        neg[dim] = -step
        offsets.append(neg)
        
        # 正方向
        pos = [0] * dimensions
        pos[dim] = step
        offsets.append(pos)
    
    return offsets

# 生成4维偏移向量，步长0.01
offsets = generate_offsets(step=0.01, dimensions=4)
for offset in offsets:
    print(offset)

def get_rail_route(start,end,trail=1):
    if trail >= 10:
        return False
    
    print(start,end)
    try:
        response = requests.get(f'https://brouter.de/brouter?lonlats={start[1]},{start[0]}|{end[1]},{end[0]}&profile=rail&format=geojson&alternativeidx=0')
        data = response.json()
        return data['features'][-1]['geometry']['coordinates']
    except:
        print('fail')
        for i in offsets:
            next_trial = get_rail_route([start[0]+i[0],start[1]+i[1]],[end[0]+i[2],end[1]+i[3]],trail=trail+1)
            if next_trial:
                return next_trial
        return False
get_rail_route(statgeo['长沙'],statgeo['株洲'])

reading station data
[-0.01, 0, 0, 0]
[0.01, 0, 0, 0]
[0, -0.01, 0, 0]
[0, 0.01, 0, 0]
[0, 0, -0.01, 0]
[0, 0, 0.01, 0]
[0, 0, 0, -0.01]
[0, 0, 0, 0.01]
[28.196468, 113.0079938] [27.83784416, 113.1503408]
fail
[28.186467999999998, 113.0079938] [27.83784416, 113.1503408]


[[113.007974, 28.186468, 38.25],
 [113.00798, 28.186546, 38.25],
 [113.007967, 28.18717, 39.0],
 [113.008025, 28.187987, 38.0],
 [113.00805, 28.188384, 37.0],
 [113.008117, 28.189385, 38.0],
 [113.008217, 28.189982, 37.75],
 [113.008169, 28.189314, 37.75],
 [113.008147, 28.188809, 37.25],
 [113.008105, 28.188383, 36.75],
 [113.008076, 28.187965, 37.75],
 [113.008054, 28.187205, 38.75],
 [113.008019, 28.186339, 38.0],
 [113.007952, 28.18535, 38.75],
 [113.0079, 28.184707, 39.0],
 [113.007169, 28.173552, 42.5],
 [113.007139, 28.173162, 43.0],
 [113.00702, 28.171319, 45.0],
 [113.006971, 28.170653, 45.5],
 [113.006962, 28.170252, 45.25],
 [113.006974, 28.169923, 45.25],
 [113.007026, 28.169619, 46.0],
 [113.00713, 28.169319, 46.5],
 [113.007259, 28.169039, 46.5],
 [113.007449, 28.168704, 45.5],
 [113.008146, 28.1677, 45.25],
 [113.008306, 28.167445, 46.0],
 [113.00842, 28.167203, 47.25],
 [113.008529, 28.166949, 48.0],
 [113.008608, 28.16668, 48.75],
 [113.008633, 28.166493, 49.0],
 [113.

In [33]:
# -*- coding: GBK -*-
import json
import pandas as pd
import folium
from folium import plugins
import datetime
import sys

database = "../../database"
output = "output"

print("reading station data")
statdata = pd.read_csv(f'{database}//stations_data.csv',encoding='gbk')
#statdata.head()
statgeo = {}
stat_affiliation = {}
for i,row in statdata.iterrows():
    lat = row['纬度']
    lng = row['经度']
    loc = [lat,lng]
#     loc = correct(loc)
    statgeo[row['车站']]=loc
    stat_affiliation[row['车站']]={'lat':lat,'lng':lng,
                                 'bureau':row['路局'],
                                 'city':row['市'],
                                 'province':row['省']}
#print(statgeo)    

print("reading train travel record")
data = pd.read_excel(f'{database}//火车乘坐记录.xlsx', sheet_name = 0)
data = data.fillna('nan')
data2 = pd.read_excel(f'{database}//火车乘坐记录.xlsx', sheet_name = '乘坐列表')
data2 = data2.fillna('nan')
print(data2)

print("drawing map")
#绘制路径
# m = folium.Map(location=statgeo['上海'], zoom_start=5)
m = folium.Map(location=statgeo['上海'],
               zoom_start=5,
               control_scale=True,
               control=False,
               tiles=None
              )

folium.TileLayer(tiles='https://webrd04.is.autonavi.com/appmaptile?lang=zh_cn&size=1&scale=1&style=7&x={x}&y={y}&z={z}',#'http://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}{r}.png',
                 attr="&copy; <a href='https://stadiamaps.com/' target='_blank'>Stadia Maps</a> &copy; <a href='https://openmaptiles.org/' target='_blank'>OpenMapTiles</a> &copy; <a href='https://www.openstreetmap.org/copyright' target='_blank'>OpenStreetMap</a>&copy; <a href='https://stamen.com/' target='_blank'>Stamen Design</a>",
                 min_zoom=0,
                 max_zoom=19,
                 control=True,
                 show=True,
                 overlay=False,
                 name='year'
                ).add_to(m)
polyline_set = []
point_set = []
heatmap_data = []

for i in range(4):
    a = data.iloc[i]
    train = a['车次']
    date = a.iloc[1]
    color = 'blue'
    #if type(date) != pd._libs.tslibs.nattype.NaTType:
    #    dt = pd.to_datetime(date)
    #    if dt.year>2024:
    #        color = 'red'
    b = data2.iloc[i]
    #if b['上车站'] == '大同南' or b['下车站'] == '大同南':
    #    color = 'blue'
    duration = int(b['时长.1'])
    distance = int(b['里程'])
    From = b['上车站']
    To = b['下车站']
    Fromcity = stat_affiliation[From]['city']
    Tocity = stat_affiliation[To]['city']
    Fromprov = stat_affiliation[From]['province']
    Toprov = stat_affiliation[To]['province']
    polyline = []
    pp = a[5]
    for p in a[5:]:
        if p == 'nan':
            continue
        if p not in statgeo:
            continue
        if p == a[5]:
            continue
        
        start = statgeo[pp]
        end = statgeo[p]
        for loc in get_rail_route(start,end):
            try:
                polyline.append([loc[1],loc[0]])
            except:
                polyline.append(statgeo[p])
                break
        pp = p
#         loc = statgeo[p]
#         name = p
#         polyline.append(loc)
#         heatmap_data.append(loc)
#     for p in get_rail_route(From,To):
#         try:
#             polyline.append([p[1],p[0]])
#         except:
#             break
    
    polyline_set.append([polyline,color,train,duration,distance,From,To,Fromcity,Tocity,Fromprov,Toprov])
    point_set.append([statgeo[To],color,To,train,duration,distance,From,To,Fromcity,Tocity,Fromprov,Toprov])
    point_set.append([statgeo[a.iloc[5]],color,a.iloc[5],train,duration,distance,From,To,Fromcity,Tocity,Fromprov,Toprov])

    
#after = folium.FeatureGroup(name='since 2024')
#before = folium.FeatureGroup(name='before 2024')
group = folium.FeatureGroup(name = 'all')
group1 = folium.FeatureGroup(name = 'discard')
for polyline,color,train,duration,distance,From,To,Fromcity,Tocity,Fromprov,Toprov in polyline_set:
    op = 0.3
    #group  = after
    if color == 'blue':
        op = 0.1
        #group = before
    geojson = {
        "type":"Feature",
        "properties":{"id":str(train),
                      "duration":duration,
                      "distance":distance,
                      "from":From,
                      "to":To,
                      "fromcity":Fromcity,
                      "tocity":Tocity,
                      "fromprov":Fromprov,
                      "toprov":Toprov},
        "geometry":{
            "type":"LineString",
            "coordinates":[[lng,lat] for lat, lng in polyline]}
    }
    #print(geojson)
    folium.GeoJson(
        geojson,
        name = train,
        color=color,
        weight=6,
        opacity=op
    ).add_to(group)

for point,color,name,train,duration,distance,From,To,Fromcity,Tocity,Fromprov,Toprov in point_set:
    #group = after
    #if color == 'blue':
        #group = before
    geojson = {
        "type":"Feature",
        "properties":{"id":str(train),
                      "duration":duration,
                      "distance":distance,
                      "from":From,
                      "to":To,
                      "fromcity":Fromcity,
                      "tocity":Tocity,
                      "fromprov":Fromprov,
                      "toprov":Toprov},
        "geometry":{
            "type":"Point",
            "coordinates":[point[1],point[0]]}
    }
    folium.GeoJson(
        geojson,
        name = train,
        popup=folium.Popup(name,max_width=10),
        marker=folium.CircleMarker(
        radius=3,  # 默认半径，会被style_function覆盖
        fill=True,
        fill_color='#FFD700',
        color=color #'#FF1493',
        )
    ).add_to(group1)

group.add_to(m)
#after.add_to(m)
#before.add_to(m)
#folium.LayerControl(collapsed=False).add_to(m)
from folium import Element
map_var = m.get_name()
custom_html = f"""
<script>

    window.addEventListener("message", function (event) {{
        const msg = event.data;
        if (msg.type === "highlight_train") {{
            highlightRoute(msg.train);
        }}
        else if (msg.type === "Travel Duration") {{
            highlightDuration(msg.start, msg.end);
        }}
        else if (msg.type === "Travel Distance") {{
            highlightDistance(msg.start, msg.end);
        }}
        else if (msg.type === "highlight_station") {{
            highlightStation(msg.station);
        }}
        else if (msg.type === "highlight_city"){{
            highlightCity(msg.city);
        }}
        else if (msg.type === "highlight_prov"){{
            highlightProv(msg.prov);
        }}
        else if (msg.type === "highlight_train_type"){{
            highlightTrainType(msg.train_type);
        }}
    }});
    function highlightTrainType(train_type) {{
        {map_var}.eachLayer(function (layer) {{
            if (layer.feature) {{
                let train = layer.feature.properties.id;
                let flag = false;
                if (/^[A-Za-z]/.test(train)) flag = ( train[0] === train_type);
                else flag = (train_type === "数字车次");
                if (flag) {{
                    layer.setStyle({{
                        color: "red",
                        weight: 6,
                        opacity: 0.7
                    }});
                }}
                else {{
                    layer.setStyle({{
                        color: "blue",
                        weight: 3,
                        opacity: 0.2
                    }});
                }};
            }}
        }});
    }}
    function highlightProv(prov) {{
        {map_var}.eachLayer(function (layer) {{
            if (layer.feature) {{
                if (layer.feature.properties.fromprov === prov || layer.feature.properties.toprov === prov) {{
                    layer.setStyle({{
                        color: "red",
                        weight: 6,
                        opacity: 0.7
                    }});
                }}
                else {{
                    layer.setStyle({{
                        color: "blue",
                        weight: 3,
                        opacity: 0.2
                    }});
                }};
            }}
        }});
    }}
    function highlightCity(city) {{
        {map_var}.eachLayer(function (layer) {{
            if (layer.feature) {{
                if (layer.feature.properties.fromcity === city || layer.feature.properties.tocity === city) {{
                    layer.setStyle({{
                        color: "red",
                        weight: 6,
                        opacity: 0.7
                    }});
                }}
                else {{
                    layer.setStyle({{
                        color: "blue",
                        weight: 3,
                        opacity: 0.2
                    }});
                }};
            }}
        }});
    }}
    function highlightStation(station) {{
        {map_var}.eachLayer(function (layer) {{
            if (layer.feature) {{
                if (layer.feature.properties.from === station || layer.feature.properties.to === station) {{
                    layer.setStyle({{
                        color: "red",
                        weight: 6,
                        opacity: 0.7
                    }});
                }}
                else {{
                    layer.setStyle({{
                        color: "blue",
                        weight: 3,
                        opacity: 0.2
                    }});
                }};
            }}
        }});
    }}
    function highlightDistance(start, end) {{
        {map_var}.eachLayer(function (layer) {{
            if (layer.feature) {{
                if (layer.feature.properties.distance >= start && layer.feature.properties.distance < end) {{
                    layer.setStyle({{
                        color: "red",
                        weight: 6,
                        opacity: 0.7
                    }});
                }}
                else {{
                    layer.setStyle({{
                        color: "blue",
                        weight: 3,
                        opacity: 0.2
                    }});
                }};
            }}
        }});
    }}
    function highlightDuration(start, end) {{
        {map_var}.eachLayer(function (layer) {{
            if (layer.feature) {{
                if (layer.feature.properties.duration >= start && layer.feature.properties.duration < end) {{
                    layer.setStyle({{
                        color: "red",
                        weight: 6,
                        opacity: 0.7
                    }});
                }}
                else {{
                    layer.setStyle({{
                        color: "blue",
                        weight: 3,
                        opacity: 0.2
                    }});
                }};
            }}
        }});
    }}
    function highlightRoute(train) {{

        {map_var}.eachLayer(function (layer) {{
            if (layer.feature) {{
                if (layer.feature.properties.id === train) {{
                    layer.setStyle({{
                        color: "red",
                        weight: 6,
                        opacity: 1
                    }});
                }}
                else {{
                    layer.setStyle({{
                        color: "blue",
                        weight: 3,
                        opacity: 0.2
                    }});
                }};
            }} 
        }});
    }}
    

</script>
"""
m.get_root().html.add_child(Element(custom_html))
m.save(f'{output}/map_rail.html')


reading station data
reading train travel record
      序号         日期     车次    始发    终到   上车站   下车站      上车时间      下车时间   席位  \
0      1        NaT  C7670   nan   nan    珠海    顺德  13:50:00  14:41:00  二等座   
1      2        NaT  C7675   nan   nan   广州南    珠海  10:55:00  12:11:00  二等座   
2      3        NaT   G302   nan   nan   广州南   北京西  08:09:00  19:13:00  二等座   
3      4        NaT   Z201   nan   nan   北京西    广州  20:24:00  18:05:00   硬卧   
4      5 2016-08-05   Z202    三亚   北京西    广州   北京西  08:50:00  06:49:00   硬卧   
..   ...        ...    ...   ...   ...   ...   ...       ...       ...  ...   
187  188 2026-05-02  G1258  上海虹桥   长春西   张家港    海安  10:30:00  11:07:00  二等座   
188  189 2026-05-02  K8393    南京    如东    海安    如东  11:31:00  12:27:00   软座   
189  190 2026-05-02  K8394    如东    南京    如东    栟茶  14:00:00  14:22:00   软座   
190  191 2026-05-16  D3142    龙岩   南京南  上海虹桥    苏州  16:01:00  16:36:00  二等座   
191  192 2026-05-16   G807   北京南  上海虹桥   苏州北  上海虹桥  23:01:00  23:25:00  二等座   

  

C:\Users\wangt\AppData\Local\Temp\ipykernel_32112\4059784985.py:80: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  pp = a[5]
C:\Users\wangt\AppData\Local\Temp\ipykernel_32112\4059784985.py:86: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if p == a[5]:


[22.23445808, 113.5202458] [22.26823072, 113.5100993]
[22.26823072, 113.5100993] [22.36321008, 113.546154]
[22.36321008, 113.546154] [22.40094432, 113.546548]
[22.40094432, 113.546548] [22.4863168, 113.5278311]
[22.4863168, 113.5278311] [22.54366096, 113.4275479]
[22.54366096, 113.4275479] [22.55772464, 113.3764212]
[22.55772464, 113.3764212] [22.60437248, 113.2787978]
[22.60437248, 113.2787978] [22.647752, 113.2640213]
[22.647752, 113.2640213] [22.71292032, 113.3029328]
[22.71292032, 113.3029328] [22.75105072, 113.3065776]
[22.75105072, 113.3065776] [22.82245856, 113.3137689]
fail
[22.741050719999997, 113.3065776] [22.82245856, 113.3137689]
fail
[22.731050719999995, 113.3065776] [22.82245856, 113.3137689]
fail
[22.721050719999994, 113.3065776] [22.82245856, 113.3137689]
fail
[22.711050719999992, 113.3065776] [22.82245856, 113.3137689]
fail
[22.70105071999999, 113.3065776] [22.82245856, 113.3137689]
fail
[22.69105071999999, 113.3065776] [22.82245856, 113.3137689]
fail
[22.6810507199999

[23.37698352, 113.1974286] [23.19108544, 113.2401819]
[23.19108544, 113.2401819] [23.14958768, 113.2518061]
fail
[23.181085439999997, 113.2401819] [23.14958768, 113.2518061]
fail
[23.171085439999995, 113.2401819] [23.14958768, 113.2518061]
fail
[23.161085439999994, 113.2401819] [23.14958768, 113.2518061]
fail
[23.151085439999992, 113.2401819] [23.14958768, 113.2518061]
fail
[23.14108543999999, 113.2401819] [23.14958768, 113.2518061]


In [39]:
import requests
import networkx as nx
import folium
from math import radians, sin, cos, sqrt, atan2
from shapely.geometry import LineString


OVERPASS_URL = "https://overpass-api.de/api/interpreter"


def overpass_query(query):
    r = requests.post(OVERPASS_URL, data=query.encode("utf-8"), timeout=120)
    r.raise_for_status()
    return r.json()


def haversine(lat1, lon1, lat2, lon2):
    R = 6371000
    phi1, phi2 = radians(lat1), radians(lat2)
    dphi = radians(lat2 - lat1)
    dlambda = radians(lon2 - lon1)

    a = sin(dphi / 2) ** 2 + cos(phi1) * cos(phi2) * sin(dlambda / 2) ** 2
    return 2 * R * atan2(sqrt(a), sqrt(1 - a))


def get_station(name):
    query = f"""
    [out:json][timeout:30];
    (
      node["railway"="station"]["name"="{name}"];
      node["public_transport"="station"]["name"="{name}"];
    );
    out body;
    """
    data = overpass_query(query)

    if not data["elements"]:
        raise ValueError(f"找不到车站：{name}")

    e = data["elements"][0]
    return {
        "name": name,
        "lat": e["lat"],
        "lon": e["lon"],
        "id": e["id"]
    }


def get_railways_between(station_a, station_b, buffer_deg=0.15):
    south = min(station_a["lat"], station_b["lat"]) - buffer_deg
    north = max(station_a["lat"], station_b["lat"]) + buffer_deg
    west = min(station_a["lon"], station_b["lon"]) - buffer_deg
    east = max(station_a["lon"], station_b["lon"]) + buffer_deg

    query = f"""
    [out:json][timeout:120];
    (
      way["railway"="rail"]({south},{west},{north},{east});
      way["railway"="light_rail"]({south},{west},{north},{east});
      way["railway"="subway"]({south},{west},{north},{east});
    );
    out body;
    >;
    out skel qt;
    """

    return overpass_query(query)


def build_graph(osm_json):
    nodes = {}
    ways = []

    for e in osm_json["elements"]:
        if e["type"] == "node":
            nodes[e["id"]] = (e["lat"], e["lon"])
        elif e["type"] == "way":
            ways.append(e)

    G = nx.Graph()

    for way in ways:
        node_ids = way.get("nodes", [])

        for u, v in zip(node_ids[:-1], node_ids[1:]):
            if u not in nodes or v not in nodes:
                continue

            lat1, lon1 = nodes[u]
            lat2, lon2 = nodes[v]
            dist = haversine(lat1, lon1, lat2, lon2)

            G.add_node(u, lat=lat1, lon=lon1)
            G.add_node(v, lat=lat2, lon=lon2)
            G.add_edge(u, v, weight=dist)

    return G


def nearest_graph_node(G, lat, lon):
    best_node = None
    best_dist = float("inf")

    for node, data in G.nodes(data=True):
        d = haversine(lat, lon, data["lat"], data["lon"])
        if d < best_dist:
            best_dist = d
            best_node = node

    return best_node, best_dist


def draw_route(G, route_nodes, station_a, station_b, output="rail_route.html"):
    center_lat = (station_a["lat"] + station_b["lat"]) / 2
    center_lon = (station_a["lon"] + station_b["lon"]) / 2

    m = folium.Map(location=[center_lat, center_lon], zoom_start=10)

    route_coords = [
        [G.nodes[n]["lat"], G.nodes[n]["lon"]]
        for n in route_nodes
    ]

    folium.PolyLine(
        route_coords,
        weight=5,
        tooltip="railway route"
    ).add_to(m)

    folium.Marker(
        [station_a["lat"], station_a["lon"]],
        popup=station_a["name"]
    ).add_to(m)

    folium.Marker(
        [station_b["lat"], station_b["lon"]],
        popup=station_b["name"]
    ).add_to(m)

    m.save(output)
    print(f"已保存到 {output}")


def main():
    station_name_a = "上海虹桥站"
    station_name_b = "上海站"

    station_a = get_station(station_name_a)
    station_b = get_station(station_name_b)

    osm_json = get_railways_between(station_a, station_b, buffer_deg=0.25)
    G = build_graph(osm_json)

    start_node, d1 = nearest_graph_node(G, station_a["lat"], station_a["lon"])
    end_node, d2 = nearest_graph_node(G, station_b["lat"], station_b["lon"])

    print("起点最近轨道距离:", d1, "米")
    print("终点最近轨道距离:", d2, "米")

    route_nodes = nx.shortest_path(G, start_node, end_node, weight="weight")

    draw_route(G, route_nodes, station_a, station_b)


main()

HTTPError: 406 Client Error: Not Acceptable for url: https://overpass-api.de/api/interpreter